# Toy Experiments

We want to perform ASR with Continuous Flow Matching. This is, predicting a sequence of categorical tokens by cross-attending to a continuous latent embedding that contains information about the whole sequence.

To do this, we want to consider the following toy experiment. Let's create a vocabulary of $|\mathcal{V}|$ tokens and try to predict sequences of L tokens.

Let's say valid sequences are the ones in which the $\sum_\ell w^\ell = 0$.

We want to condition on $a^\ell = e_{w^\ell} + \eta \, \varepsilon^\ell, \varepsilon \sim \mathcal{N}(0, I)$ with $\eta$ controlling the noise level.

In [29]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

In [47]:
vocab_size = 8
seq_length = 6
num_samples = 100
num_valid_transitions = 3

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# row indices: each row repeated k times -> shape (V, k)
row_idx = torch.arange(vocab_size, device=device).unsqueeze(1).expand(-1, num_valid_transitions)

# distinct successors per row, sampled without replacement
col_idx = torch.stack([
    torch.randperm(vocab_size)[:num_valid_transitions]
    for _ in range(vocab_size)
])

transition_matrix = torch.zeros((vocab_size, vocab_size), dtype=torch.int32, device=device)
transition_matrix[row_idx, col_idx] = 1

# each row has exactly k successors
assert (transition_matrix.sum(dim=1) == num_valid_transitions).all()

# exact count of valid length-L sequences, matches your sampler's support
def count_valid(trans_matrix, seq_length):
    ones = torch.ones(trans_matrix.shape[0], device=trans_matrix.device)
    M = torch.linalg.matrix_power(trans_matrix, seq_length - 1)
    return int((ones @ M @ ones).item())

valid_seq_count = count_valid(transition_matrix.float(), seq_length=seq_length)

print(
    f"Valid sequences of length {seq_length}: {valid_seq_count}"
)

# row-normalize for sampling
prob_trans_matrix = transition_matrix.float() / transition_matrix.sum(dim=1, keepdim=True)

Valid sequences of length 6: 1944


In [49]:
def sample_sequence(
    trans_matrix,
    sample_size,
    seq_length,
    device=None,
):
    device = device or trans_matrix.device
    vocab_size = trans_matrix.shape[0]

    # Uniform distribution
    first_token_dist = torch.ones(vocab_size, device=device) / vocab_size

    # Sample first token
    w = torch.zeros(sample_size, seq_length, dtype=torch.long, device=device)
    w[:, 0] = torch.multinomial(first_token_dist, sample_size, replacement=True)

    # Fill the rest of the sequence
    for l in range(1, seq_length):
        probs = trans_matrix[w[:, l - 1]]
        w[:, l] = torch.multinomial(probs, 1).squeeze(1)

    return w

all_sequences = torch.empty(0, seq_length, dtype=torch.long, device=device)

while all_sequences.shape[0] < valid_seq_count:
    new_samples = sample_sequence(prob_trans_matrix, num_samples, seq_length)
    all_sequences = torch.cat([all_sequences, new_samples], dim=0)
    all_sequences = torch.unique(all_sequences, dim=0)

In [50]:
sigma = 0.4

def check_valid(sequences, transition_matrix):
    # Cut first and last tokens to get pairs of transitions
    prev = sequences[:, :-1]
    nxt  = sequences[:, 1:]

    # entry [b, l] = 1 iff (prev[b,l] -> nxt[b,l]) is a legal transition
    legal = transition_matrix[prev, nxt].bool()

    per_sequence_valid = legal.all(dim=1)   # (B,) — every transition must hold
    num_valid = per_sequence_valid.sum().item()
    frac_valid = per_sequence_valid.float().mean().item()

    return num_valid, frac_valid, per_sequence_valid


def make_noisy_ids(sequences, vocab_size, sigma):
    onehot = F.one_hot(sequences, num_classes=vocab_size).float()
    noisy = onehot + sigma * torch.randn_like(onehot)
    return noisy.argmax(dim=-1)

noisy_samples = make_noisy_ids(all_sequences, vocab_size=transition_matrix.shape[0], sigma=sigma)
num_valid, frac_valid, per_sequence_valid = check_valid(noisy_samples, transition_matrix)

print(
    f"Number of valid samples: {num_valid} "
    f"out of {noisy_samples.size(0)} ({frac_valid * 100:.2f}%)"
)

Number of valid samples: 811 out of 1944 (41.72%)


In [51]:

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):  # t: (B,)
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t[:, None] * freqs[None, :]  # (B, half)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, dim)


class DenoiserBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # bidirectional self-attention — no causal mask, the model sees all positions
        h = self.norm1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]
        x = x + self.ff(self.norm2(x))
        return x

class Denoiser(nn.Module):
    def __init__(self, vocab_size, seq_len, d_model=128, n_heads=4, n_layers=3, d_ff=256):
        super().__init__()
        self.x_proj = nn.Linear(vocab_size, d_model)
        self.a_proj = nn.Linear(vocab_size, d_model)
        self.time_embed = SinusoidalTimeEmbedding(d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        self.blocks = nn.ModuleList([DenoiserBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x_t, t, a):
        # x_t: (B, L, V), t: (B,), a: (B, L, V)
        h = self.x_proj(x_t) + self.a_proj(a) + self.pos_embed
        h = h + self.time_embed(t)[:, None, :]   # broadcast time to every position
        for block in self.blocks:
            h = block(h)
        return self.head(self.norm_out(h))       # (B, L, V) logits — D_hat pre-softmax

In [56]:
def make_epoch_loader(sequences, batch_size, shuffle):
    return torch.utils.data.DataLoader(sequences, batch_size=batch_size, shuffle=shuffle)


def run_epoch(model, loader, transition_matrix, device, optimizer=None):
    """One pass over `loader`. If optimizer is given, trains; otherwise evaluates.
    Returns a dict of epoch-averaged metrics."""
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss, total_seqs = 0.0, 0
    total_valid, total_unique_valid = 0, 0

    ctx = torch.enable_grad() if train_mode else torch.no_grad()
    with ctx:
        for batch in loader:
            batch = batch.to(device)
            B = batch.size(0)
            vocab_size = model.head.out_features

            x1 = F.one_hot(batch, vocab_size).float()
            x0 = torch.randn_like(x1)
            t = torch.rand(B, device=device)
            I_t = (1 - t)[:, None, None] * x0 + t[:, None, None] * x1
            a = x1 + sigma * torch.randn_like(x1)

            logits = model(I_t, t, a)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), batch.reshape(-1))

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            pred_sequences = logits.argmax(dim=-1)
            num_valid, _, valid_mask = check_valid(pred_sequences, transition_matrix)
            num_unique_valid = torch.unique(pred_sequences[valid_mask], dim=0).size(0) if num_valid > 0 else 0

            total_loss += loss.item() * B
            total_seqs += B
            total_valid += num_valid
            total_unique_valid += num_unique_valid

    return {
        "loss": total_loss / total_seqs,
        "frac_valid": total_valid / total_seqs,
        "unique_valid_count": total_unique_valid,   # not a fraction across batches — see note below
    }

In [ ]:
num_epochs = 50
bs = 32
sigma = 0.5

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

train_size = int(0.8 * all_sequences.size(0))
train_data = all_sequences[:train_size]
val_data = all_sequences[train_size:]

train_loader = make_epoch_loader(train_data, bs, shuffle=True)
val_loader = make_epoch_loader(val_data, bs, shuffle=False)

model = Denoiser(vocab_size=vocab_size, seq_len=seq_length).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

history = {"train": [], "val": []}

for epoch in range(num_epochs):
    train_metrics = run_epoch(model, train_loader, transition_matrix, device, optimizer)
    val_metrics = run_epoch(model, val_loader, transition_matrix, device, optimizer=None)

    history["train"].append(train_metrics)
    history["val"].append(val_metrics)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"train loss {train_metrics['loss']:.4f} valid% {train_metrics['frac_valid']*100:.1f} | "
        f"val loss {val_metrics['loss']:.4f} valid% {val_metrics['frac_valid']*100:.1f}"
    )

Epoch 1/50 | train loss 0.9414 valid% 32.8 | val loss 0.7303 valid% 35.0
Epoch 2/50 | train loss 0.6472 valid% 42.4 | val loss 0.6933 valid% 39.3
Epoch 3/50 | train loss 0.6214 valid% 44.8 | val loss 0.6451 valid% 45.5
Epoch 4/50 | train loss 0.5736 valid% 48.0 | val loss 0.7183 valid% 41.9
Epoch 5/50 | train loss 0.5138 valid% 53.9 | val loss 0.6372 valid% 46.8
Epoch 6/50 | train loss 0.4827 valid% 54.2 | val loss 0.6420 valid% 47.8
Epoch 7/50 | train loss 0.4972 valid% 53.6 | val loss 0.6451 valid% 49.1
Epoch 8/50 | train loss 0.5156 valid% 52.7 | val loss 0.5890 valid% 51.2
Epoch 9/50 | train loss 0.4999 valid% 53.5 | val loss 0.6182 valid% 47.0
Epoch 10/50 | train loss 0.4782 valid% 54.1 | val loss 0.5895 valid% 46.5
Epoch 11/50 | train loss 0.5000 valid% 54.0 | val loss 0.6201 valid% 46.3
Epoch 12/50 | train loss 0.4915 valid% 53.2 | val loss 0.6869 valid% 43.4
Epoch 13/50 | train loss 0.4885 valid% 55.5 | val loss 0.5970 valid% 49.9
Epoch 14/50 | train loss 0.4785 valid% 55.6 | v

In [44]:
logits.argmax(dim=-1).shape

torch.Size([32, 6])